# Forecasting two-stream plasma dynamics

This notebook runs on the **included processed data**. It needs no WarpX
installation, raw particle files or Google Drive mount. A CPU runtime is enough.

The five runs share physical parameters but have different random initial
particle samples. We forecast the next pair of position-velocity maps.
Seeds 1-3 train the models, seed 4 selects settings, and seed 5 is the test.
This is an initial one-step comparison, not a completed numerical-validation
or long-rollout study.

In [ ]:
# Locate a local checkout, or retrieve the repository in a fresh Colab runtime.
import os
import sys
import subprocess
import importlib.util
from pathlib import Path

REPOSITORY = "https://github.com/jamothemadlad/2Stream.git"
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "data/manifest.json").is_file()), None)
if ROOT is None:
    ROOT = Path("/content/2Stream") if Path("/content").exists() else Path.cwd() / "2Stream"
    if not ROOT.exists():
        subprocess.check_call(["git", "clone", REPOSITORY, str(ROOT)])
    if not (ROOT / "data/manifest.json").is_file():
        raise RuntimeError(f"The checkout at {ROOT} does not contain the processed data.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Use Colab's installed scientific packages; install only missing dependencies.
packages = {"numpy": "numpy>=1.26,<3", "scipy": "scipy>=1.11,<2",
            "matplotlib": "matplotlib>=3.7,<4", "sklearn": "scikit-learn>=1.5,<2"}
missing = [requirement for module, requirement in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib_cache"))
print("Project:", ROOT)
print("Python:", sys.version.split()[0])

## 1 Load and verify the data

Each `phase_mass` array has axes **time, original stream, position, velocity**.
The velocity axis contains physical `v/c`, obtained from SI momentum.
The map contains normalised weighted bin mass; each stream sums to one.
The first check verifies the included files against their processed checksums.

In [ ]:
import json
import hashlib
import importlib.metadata
import numpy as np
import matplotlib.pyplot as plt
from twostream.forecast import load_runs, run_study, plot_example

manifest_path = ROOT / "data/manifest.json"
manifest = json.loads(manifest_path.read_text())
for item in manifest["runs"]:
    file = manifest_path.parent / item["file"]
    assert hashlib.sha256(file.read_bytes()).hexdigest() == item["processed_sha256"]
runs = load_runs(ROOT / "data/processed")
for seed, run in runs.items():
    print(f"Seed {seed}: maps {run['phase_mass'].shape}; "
          f"time {run['times_s'][0]:.3g}-{run['times_s'][-1]*1e9:.3f} ns; "
          f"excluded weight {run['excluded_weight_fraction'].max():.2g}")
versions = {name: importlib.metadata.version(name)
            for name in ["numpy", "scipy", "matplotlib", "scikit-learn"]}
print("Package versions:", versions)

## 2 Look at a distribution

These are arrays, not screenshots supplied to the model. The colour is weighted
bin mass. Both axes and the bins are shared across all runs.

In [ ]:
run = runs[1]
frame = len(run["times_s"]) // 2
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
extent = [*run["z_edges_m"][[0, -1]], *run["velocity_edges_over_c"][[0, -1]]]
for stream, ax in enumerate(axes):
    im = ax.imshow(run["phase_mass"][frame, stream].T, origin="lower",
                   aspect="auto", extent=extent, cmap="viridis")
    ax.set(title=str(run["species"][stream]), xlabel="Position z (m)")
    fig.colorbar(im, ax=ax, label="Normalised bin mass")
axes[0].set_ylabel("Physical velocity / c")
fig.suptitle(f"Seed 1 at iteration {run['iterations'][frame]}")
fig.tight_layout()
plt.show()

## 3 Fit PCA and compare forecasters

The example uses 32 components and three earlier snapshots. PCA and input
scaling are fitted on seeds 1-3 only. Both regression families receive the
same past-state information, including past Ez profiles. Seed 4 chooses
regularisation and tree settings; seed 5 is not used in that selection.

The **PCA reconstruction diagnostic** uses the true future map only to measure
compression error. It is not a forecast and is never an input to a model.
Predictions are scored before any positivity or particle-number correction.

In [ ]:
study = run_study(runs, train=(1, 2, 3), validation=(4,), test=(5,),
                  components=32, history=3)
report = study["report"]
print("Split:", report["split"])
print("PCA training variance retained:",
      f"{100*report['pca_training_explained_variance_fraction']:.2f}%")
print("Selected linear alpha:", report["linear_selection"]["alpha"])
print("Selected forest:", {k:v for k,v in report["random_forest_selection"].items() if k != "trials"})
print(f"{'Method':36s} {'relative L1':>12s} {'relative L2':>12s} {'negative mass':>14s}")
for row in report["test_scores"]:
    print(f"{row['method']:36s} {row['mean_relative_L1']:12.4f} "
          f"{row['mean_relative_L2']:12.4f} {row['mean_negative_mass_fraction']:14.4g}")

## 4 Inspect an actual held-out prediction

A low numerical error is not sufficient on its own. Compare the structures,
locations and velocity spread, and inspect where the difference is largest.
The initial local run did **not** beat persistence or ballistic transport.
The PCA reconstruction error was already large, so the representation needs
investigation alongside the forecasting models. Seed 5 has now been examined;
future model development needs validation runs and a fresh final assessment.

In [ ]:
figure = plot_example(runs[5], study["predictions"][5])
plt.show()

## 5 Save the outputs

The notebook is versioned in Git. Generated local outputs go to an ignored
directory. In a temporary Colab runtime, download anything you want to retain.

In [ ]:
output = ROOT / "outputs/notebook"
output.mkdir(parents=True, exist_ok=True)
report["environment"] = versions
(output / "metrics.json").write_text(json.dumps(report, indent=2) + "\n")
figure.savefig(output / "seed_5_forecast.png", dpi=150)
print("Saved:", output)

## What remains to investigate

- More independent test runs or a properly nested whole-seed evaluation.
- Sensitivity to histogram resolution, PCA rank and input history, selected
  using validation runs.
- Whether the models beat persistence and ballistic transport for a useful
  forecast interval, including the nonlinear part of the evolution.
- Closed-loop forecasts need a consistent way to evolve or reconstruct the
  fields. This notebook uses observed history for one-step testing.
- Numerical convergence of the reference simulation remains separate from
  passing the data-integrity checks.

The raw five-run data had no archived full per-run inputs, and openPMD's
auxiliary `dt` and boundary labels have known qualifications. See the manifest,
audit summary and README. Do not interpret this notebook as proof that those
physical-validation questions are resolved.